In [1]:
import numpy as np

from scipy.stats import norm
from scipy.spatial import cKDTree

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel,
    Matern,
    WhiteKernel
)

# ============================================================
# FUNCTION 4 - WEEK 10 BAYESIAN OPTIMISATION
# Run from inside the week10/ folder
# ============================================================
#
# Strategy:
# - Refit ARD Matern GP including Week 9.
# - Check Week 9 calibration.
# - Centre search on the NEW ACTUAL incumbent.
# - Use local + moderate-wide candidate pools.
# - Compare EI, posterior mean and UCB.
# - Avoid aggressive global exploration unless strongly supported.
# ============================================================


# ------------------------------------------------------------
# 1. Load Week 10 cumulative data
# ------------------------------------------------------------

X = np.load("function4/initial_inputs.npy")
Y = np.load("function4/initial_outputs.npy").reshape(-1)

best_idx = np.argmax(Y)
best_x = X[best_idx]
best_y = Y[best_idx]

print("================================")
print("DATA")
print("================================")

print("X shape:", X.shape)
print("Y shape:", Y.shape)

print("\nCurrent best:")
print(best_x, "->", best_y)

print("\nY range:")
print("min =", Y.min())
print("max =", Y.max())
print("std =", Y.std())


# ------------------------------------------------------------
# 2. WEEK 9 CALIBRATION CHECK
# ------------------------------------------------------------

week9_pred_mean = 0.3502398468525616
week9_pred_std = 0.26100386108191
week9_actual = 0.6704983808455585

week9_error = (
    week9_actual
    - week9_pred_mean
)

week9_z_error = (
    week9_error
    / week9_pred_std
)

print("\n================================")
print("WEEK 9 CALIBRATION CHECK")
print("================================")

print("Predicted mean:", week9_pred_mean)
print("Predicted std :", week9_pred_std)
print("Actual        :", week9_actual)

print("\nPrediction error:")
print(week9_error)

print("\nError / predicted std:")
print(week9_z_error)


# ------------------------------------------------------------
# 3. Fit ARD Matern GP
# ------------------------------------------------------------

kernel = (
    ConstantKernel(
        1.0,
        constant_value_bounds=(1e-3, 1e3)
    )
    *
    Matern(
        length_scale=np.ones(4) * 0.2,
        length_scale_bounds=(0.01, 2.0),
        nu=2.5
    )
    +
    WhiteKernel(
        noise_level=1e-5,
        noise_level_bounds=(1e-8, 1e-1)
    )
)

gp = GaussianProcessRegressor(
    kernel=kernel,
    normalize_y=True,
    n_restarts_optimizer=30,
    random_state=42
)

gp.fit(X, Y)

print("\n================================")
print("GP FIT")
print("================================")

print("\nFitted kernel:")
print(gp.kernel_)

lengthscales = gp.kernel_.k1.k2.length_scale

inverse_ls = 1.0 / lengthscales

relative_sensitivity = (
    inverse_ls
    / inverse_ls.sum()
)

print("\nARD lengthscales:")
print(lengthscales)

print(
    "\nNormalised inverse-lengthscale sensitivity:"
)
print(relative_sensitivity)


# ------------------------------------------------------------
# 4. Expected Improvement
# ------------------------------------------------------------

def expected_improvement(
    mu,
    sigma,
    best_y,
    xi=0.0
):

    improvement = (
        mu - best_y - xi
    )

    valid = sigma > 1e-12

    Z = np.zeros_like(mu)

    Z[valid] = (
        improvement[valid]
        / sigma[valid]
    )

    EI = np.zeros_like(mu)

    EI[valid] = (
        improvement[valid]
        * norm.cdf(Z[valid])
        +
        sigma[valid]
        * norm.pdf(Z[valid])
    )

    return EI


# ------------------------------------------------------------
# 5. Candidate generation
# ------------------------------------------------------------
#
# Week 9 local refinement worked very well, so keep the search
# concentrated around the new incumbent.
#
# Still include a moderate wider pool for controlled exploration.
# ------------------------------------------------------------

rng = np.random.default_rng(42)

local_scale = np.clip(
    0.20 * lengthscales,
    0.015,
    0.08
)

wide_scale = np.clip(
    0.40 * lengthscales,
    0.04,
    0.15
)

print("\n================================")
print("CANDIDATE SCALES")
print("================================")

print("Local widths:")
print(local_scale)

print("\nWide widths:")
print(wide_scale)


local_candidates = (
    best_x
    + rng.normal(
        0,
        local_scale,
        size=(120000, 4)
    )
)

wide_candidates = (
    best_x
    + rng.normal(
        0,
        wide_scale,
        size=(100000, 4)
    )
)

global_candidates = rng.uniform(
    0,
    1,
    size=(80000, 4)
)

local_candidates = np.clip(
    local_candidates,
    0,
    1
)

wide_candidates = np.clip(
    wide_candidates,
    0,
    1
)

candidates = np.vstack([
    local_candidates,
    wide_candidates,
    global_candidates
])


# ------------------------------------------------------------
# 6. Remove near-duplicates
# ------------------------------------------------------------

tree = cKDTree(X)

distance, _ = tree.query(
    candidates,
    k=1
)

candidates = candidates[
    distance > 0.008
]

print("\nCandidates after duplicate filtering:")
print(len(candidates))


# ------------------------------------------------------------
# 7. GP predictions
# ------------------------------------------------------------

mu, sigma = gp.predict(
    candidates,
    return_std=True
)


# ------------------------------------------------------------
# 8. Primary EI
# ------------------------------------------------------------

EI = expected_improvement(
    mu,
    sigma,
    best_y,
    xi=0.0
)

ei_idx = np.argmax(EI)

print("\n================================")
print("PRIMARY EI")
print("================================")

print("candidate =", candidates[ei_idx])
print("mean =", mu[ei_idx])
print("std =", sigma[ei_idx])
print("EI =", EI[ei_idx])


# ------------------------------------------------------------
# 9. EI sensitivity
# ------------------------------------------------------------

y_scale = np.std(Y)

xi_values = [
    0.0,
    0.01 * y_scale,
    0.05 * y_scale,
    0.10 * y_scale
]

print("\n================================")
print("EI SENSITIVITY")
print("================================\n")

for xi in xi_values:

    EI_test = expected_improvement(
        mu,
        sigma,
        best_y,
        xi=xi
    )

    idx = np.argmax(EI_test)

    print(
        "xi =", f"{xi:.6e}",
        "\n candidate =", candidates[idx],
        "\n mean =", round(mu[idx], 6),
        "\n std =", round(sigma[idx], 6),
        "\n EI =", round(EI_test[idx], 8),
        "\n"
    )


# ------------------------------------------------------------
# 10. Highest predicted mean
# ------------------------------------------------------------

mean_idx = np.argmax(mu)

print("\n================================")
print("HIGHEST PREDICTED MEAN")
print("================================")

print("candidate =", candidates[mean_idx])
print("mean =", mu[mean_idx])
print("std =", sigma[mean_idx])


# ------------------------------------------------------------
# 11. UCB diagnostics
# ------------------------------------------------------------

print("\n================================")
print("UCB DIAGNOSTICS")
print("================================\n")

for beta in [
    0.05,
    0.1,
    0.25,
    0.5,
    1.0
]:

    UCB = (
        mu
        + beta * sigma
    )

    idx = np.argmax(UCB)

    print(
        f"beta={beta}",
        "\n candidate =", candidates[idx],
        "\n mean =", round(mu[idx], 6),
        "\n std =", round(sigma[idx], 6),
        "\n UCB =", round(UCB[idx], 6),
        "\n"
    )


# ------------------------------------------------------------
# 12. Distance from current best
# ------------------------------------------------------------

def distance_from_best(x):
    return np.linalg.norm(
        x - best_x
    )

print("\n================================")
print("DISTANCE FROM CURRENT BEST")
print("================================")

print(
    "EI:",
    distance_from_best(
        candidates[ei_idx]
    )
)

print(
    "Highest mean:",
    distance_from_best(
        candidates[mean_idx]
    )
)

for beta in [
    0.05,
    0.1,
    0.25,
    0.5,
    1.0
]:

    UCB = mu + beta * sigma
    idx = np.argmax(UCB)

    print(
        f"UCB beta={beta}:",
        distance_from_best(
            candidates[idx]
        )
    )


# ------------------------------------------------------------
# 13. Boundary-style diagnostic
# ------------------------------------------------------------
#
# This checks whether candidates are being pushed close to
# the domain boundaries, which would suggest uncertainty-driven
# extrapolation rather than local refinement.
# ------------------------------------------------------------

def domain_boundary_status(
    x,
    tol=0.01
):

    status = []

    for j in range(len(x)):

        if x[j] <= tol:
            status.append(
                f"x{j+1}~0"
            )

        elif x[j] >= 1.0 - tol:
            status.append(
                f"x{j+1}~1"
            )

    if not status:
        return "interior"

    return ", ".join(status)


print("\n================================")
print("DOMAIN BOUNDARY CHECK")
print("================================")

print(
    "EI:",
    domain_boundary_status(
        candidates[ei_idx]
    )
)

print(
    "Highest mean:",
    domain_boundary_status(
        candidates[mean_idx]
    )
)

for beta in [
    0.05,
    0.1,
    0.25,
    0.5,
    1.0
]:

    UCB = mu + beta * sigma
    idx = np.argmax(UCB)

    print(
        f"UCB beta={beta}:",
        domain_boundary_status(
            candidates[idx]
        )
    )

DATA
X shape: (39, 4)
Y shape: (39,)

Current best:
[0.361985 0.41201  0.421437 0.427725] -> 0.6704983808455585

Y range:
min = -32.625660215962455
max = 0.6704983808455585
std = 9.561389821989987

WEEK 9 CALIBRATION CHECK
Predicted mean: 0.3502398468525616
Predicted std : 0.26100386108191
Actual        : 0.6704983808455585

Prediction error:
0.32025853399299686

Error / predicted std:
1.227026039635832

GP FIT

Fitted kernel:
2.58**2 * Matern(length_scale=[1.66, 1.44, 1.36, 1.46], nu=2.5) + WhiteKernel(noise_level=0.000633)

ARD lengthscales:
[1.65790696 1.4350531  1.36466766 1.46200228]

Normalised inverse-lengthscale sensitivity:
[0.22201647 0.2564941  0.26972329 0.25176613]

CANDIDATE SCALES
Local widths:
[0.08 0.08 0.08 0.08]

Wide widths:
[0.15 0.15 0.15 0.15]

Candidates after duplicate filtering:
299991

PRIMARY EI
candidate = [0.37167674 0.4037974  0.42658414 0.41934237]
mean = 0.4006457224950388
std = 0.27362546884175976
EI = 0.023402133537335862

EI SENSITIVITY

xi = 0.00000

In [2]:
# ============================================================
# FINAL FUNCTION 4 - WEEK 10 SELECTION
# ============================================================
#
# Week 9 was reasonably calibrated (~ +1.23 sigma) and
# produced a new observed best.
#
# EI, posterior mean and all UCB settings remain extremely
# close to the new incumbent and safely inside the domain.
#
# beta = 0.5 is selected because it introduces modest
# exploration while sacrificing almost no predicted mean.
#
# beta = 0.25 and beta = 0.5 identify the same candidate,
# giving additional stability to the selection.

beta = 0.5

UCB = mu + beta * sigma
final_idx = np.argmax(UCB)

week10_candidate = candidates[final_idx]

print("Week 10 Function 4 candidate:")
print(week10_candidate)

print("\nPredicted mean:")
print(mu[final_idx])

print("\nPredicted std:")
print(sigma[final_idx])

print("\nUCB:")
print(UCB[final_idx])

print("\nDistance from current best:")
print(
    np.linalg.norm(
        week10_candidate - best_x
    )
)

portal = "-".join(
    f"{x:.6f}"
    for x in week10_candidate
)

print("\nPortal format:")
print(portal)

Week 10 Function 4 candidate:
[0.36684455 0.40962296 0.43083265 0.41965779]

Predicted mean:
0.40410514126026165

Predicted std:
0.2668780416589186

UCB:
0.537544162089721

Distance from current best:
0.013515593694284246

Portal format:
0.366845-0.409623-0.430833-0.419658
